In [8]:
import pandas as pd
import random

# dummy categories and scenarios
scenarios = [
    {"detail": "Victim reported unauthorized transaction from bank account.", "root": "Phishing link click", "action": "Enable 2FA and block card"},
    {"detail": "Social media account hacked and personal photos leaked.", "root": "Weak password / No MFA", "action": "Account recovery and cyber cell reporting"},
    {"detail": "Harassment through continuous phone calls from unknown numbers.", "root": "Data breach from third party app", "action": "Number blocking and legal notice"},
    {"detail": "Online shopping fraud, product not delivered after payment.", "root": "Fake website / No verification", "action": "Report website and refund claim"},
    {"detail": "Blackmail using morphed images on messaging apps.", "root": "Social engineering", "action": "Psychological counseling and police complaint"}
]

data = []

for i in range(1, 101):
    scenario = random.choice(scenarios)
    case_no = f"CASE-2026-{1000 + i}"
    victim_id = f"VIC-{5000 + i}"

    data.append({
        "Case_No": case_no,
        "Victim_ID": victim_id,
        "Details": scenario["detail"],
        "Root_Cause": scenario["root"],
        "Recommended_Action": scenario["action"]
    })

# DataFrme
df = pd.DataFrame(data)

# CSV save
df.to_csv('victim_data.csv', index=False)
print("CSV File Generated Successfully with 100 rows!")

CSV File Generated Successfully with 100 rows!


In [9]:
!pip install pandas sentence-transformers hdbscan umap-learn sklearn

  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [10]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import hdbscan
import umap
import numpy as np

# 1. Data Load
df = pd.read_csv('victim_data.csv')

# 2. Vectorization (Semantic Layer)
# 'all-MiniLM-L6-v2' model fast and accurate
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df['Details'].tolist(), show_progress_bar=True)

# 3. Dimension Reduction (Optional but improves clustering)
# Unstructured text high dimensional data to organize  UMAP used
reducer = umap.UMAP(n_neighbors=15, n_components=5, metric='cosine', random_state=42)
reduced_embeddings = reducer.fit_transform(embeddings)

# 4. Intelligent Clustering (The Logic)
# HDBSCAN auto-calculate how many cluster we need
clusterer = hdbscan.HDBSCAN(min_cluster_size=5, metric='euclidean', cluster_selection_method='eom')
df['Cluster'] = clusterer.fit_predict(reduced_embeddings)

# 5. Result Analysis
print(f"Total Clusters Found: {df['Cluster'].nunique()}")
print(df[['Case_No', 'Details', 'Cluster']].head(10))

# Save the processed data
df.to_csv('processed_victim_data.csv', index=False)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Total Clusters Found: 5
          Case_No                                            Details  Cluster
0  CASE-2026-1001  Victim reported unauthorized transaction from ...        3
1  CASE-2026-1002  Blackmail using morphed images on messaging apps.        2
2  CASE-2026-1003  Online shopping fraud, product not delivered a...        4
3  CASE-2026-1004  Online shopping fraud, product not delivered a...        4
4  CASE-2026-1005  Social media account hacked and personal photo...        1
5  CASE-2026-1006  Blackmail using morphed images on messaging apps.        2
6  CASE-2026-1007  Harassment through continuous phone calls from...        0
7  CASE-2026-1008  Harassment through continuous phone calls from...        0
8  CASE-2026-1009  Blackmail using morphed images on messaging apps.        2
9  CASE-2026-1010  Social media account hacked and personal photo...        1


In [11]:
def get_victim_history(case_no):
    case_info = df[df['Case_No'] == case_no]
    if not case_info.empty:
        cluster_id = case_info['Cluster'].values[0]
        # Same cluster-er onno case gulo history hishebe dekhabe
        history = df[df['Cluster'] == cluster_id]
        recommendation = case_info['Recommended_Action'].values[0]
        return history, recommendation
    return None, None

# Test Search
history, action = get_victim_history('CASE-2026-1005')
print(f"Recommended Action for Future: {action}")

Recommended Action for Future: Account recovery and cyber cell reporting


In [12]:
import pandas as pd


df = pd.read_csv('processed_victim_data.csv')

def test_case_search(case_id):

    result = df[df['Case_No'] == case_id]

    if not result.empty:
        print(f"--- Case Details for: {case_id} ---")
        print(f"Details: {result['Details'].values[0]}")
        print(f"Root Cause: {result['Root_Cause'].values[0]}")
        print(f"Recommended Action: {result['Recommended_Action'].values[0]}")
        print("-" * 30)
    else:
        print(f"Error: Case No '{case_id}' found hoyni! Data check koro.")

# Test koro (tomar dummy data theke ekta Case No daw)
test_case_search('CASE-2026-1005')

--- Case Details for: CASE-2026-1005 ---
Details: Social media account hacked and personal photos leaked.
Root Cause: Weak password / No MFA
Recommended Action: Account recovery and cyber cell reporting
------------------------------


In [13]:
while True:
    user_input = input("\nEnter Case No (Example: CASE-2026-1010) or 'exit' to stop: ")

    if user_input.lower() == 'exit':
        break

    test_case_search(user_input)


Enter Case No (Example: CASE-2026-1010) or 'exit' to stop: CASE-2026-1010
--- Case Details for: CASE-2026-1010 ---
Details: Social media account hacked and personal photos leaked.
Root Cause: Weak password / No MFA
Recommended Action: Account recovery and cyber cell reporting
------------------------------

Enter Case No (Example: CASE-2026-1010) or 'exit' to stop: exit
